<a href="https://colab.research.google.com/github/simoambr/Alpaca_Trading/blob/main/Alpaca_Run_Comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# %% [CELL 1] ================================================================
# SETUP — installs, imports, config. Run once.
# =============================================================================
import subprocess, sys
for pkg in ("statsmodels", "pyarrow"):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])

import glob, io, os, re, time, zipfile
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests

# ---- Google Drive (Colab only; no-op elsewhere) ---------------------------
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    pass  # not running in Colab -- assume paths below are already reachable

# ---- paths ------------------------------------------------------------
# BASE_DIR is the parent folder in your Drive. SOURCE_FOLDERS is the list of
# sub-folders (each expected to contain .zip files, non-recursive) to pull
# data from -- add or remove folder names here to include/exclude a run
# without touching anything else. Order doesn't matter.
BASE_DIR = "/content/drive/MyDrive/Colab/Alpaca_Experiment_Runs"
SOURCE_FOLDERS = [
    "run_2023",
    "run_2024",
    "run_2025",
    # "run_2024_backtest_v2",   # <- example: add a folder like this to include it
]

MASTER_DIR = "/content/master_cache"       # compact Parquet "source of truth"
OUT_XLSX = "/content/combo_analysis.xlsx"  # the human-facing analysis workbook
os.makedirs(MASTER_DIR, exist_ok=True)


def gather_zip_paths(base_dir, subfolders):
    """Collect *.zip files from each named sub-folder of base_dir (non-recursive).
    Prints a per-folder count so a typo'd/missing folder is obvious immediately."""
    all_paths = []
    for folder in subfolders:
        folder_path = os.path.join(base_dir, folder)
        found = sorted(glob.glob(os.path.join(folder_path, "*.zip")))
        print(f"  {folder}: {len(found)} zip(s)" + ("" if found else "  <-- none found, check the path/name"))
        all_paths.extend(found)
    return all_paths

# ---- knobs ----------------------------------------------------------------
PARAM_COLS = ["orb_minutes", "use_volume_filter", "volume_mult", "use_vwap_filter",
              "vwap_slope_lookback", "candle_strength_pct", "use_candle_filter",
              "stop_atr_mult", "target_mult", "vwap_min_slope", "use_direction_bias"]

FDR_ALPHA = 0.05          # significance threshold for FDR-corrected p-values
ROBUSTNESS_MIN = 0.55     # min fraction of profitable days for the Shortlist
TOP_N = 30                # size of the Top Winners / Top Losers tabs
IQR_K = 1.5               # low-outlier fence multiplier on "periods tested"

# equal-weight Balance ("Composite") Score components: metric -> "higher"/"lower" is better
BALANCE_COMPONENTS = {
    "Robustness (% Profitable Days)": "higher",
    "Total (Pnl)": "higher",
    "Profit Factor": "higher",
    "Expectancy ($/trade)": "higher",
    "Average (Pnl)": "higher",
    "Error (stdev/avg %)": "lower",
}

PATH_RE = re.compile(r"sweep_results/([^/]+)/([^/]+)\.csv$")
FOOTER_RESULT_COLS = {"stopped_out", "hit_target", "closed_eod", "win_rate_%", "total_net_pnl_$"}
# trades header: symbol,bias,outcome,entry_time,entry,stop,target,stop_mult,target_mult,exit_time,exit,shares,pnl,r_multiple
SYMBOL_IDX, ENTRY_IDX, SHARES_IDX, PNL_IDX = 0, 4, 11, 12

print("Config loaded.")


# %% [CELL 2] ================================================================
# PARSING — one pass per zip (multi-process), pure string splitting (no
# per-file pandas.read_csv). Produces two SMALL aggregate tables, never a
# per-trade table, which is what keeps this feasible at 100k-1M+ files:
#   combo_days        : one row per (combo, date)              -> Combo Metrics
#   combo_symbol_days : one row per (combo, symbol, date)      -> Symbol Stats
# =============================================================================
def _cast(val):
    val = val.strip()
    if val in ("True", "False"):
        return val == "True"
    try:
        f = float(val)
        return int(f) if f.is_integer() else f
    except ValueError:
        return val


def _parse_one_zip(zip_path):
    combo_day_rows, combo_symbol_day_rows = [], []
    with zipfile.ZipFile(zip_path) as zf:
        for name in zf.namelist():
            m = PATH_RE.search(name)
            if not m:
                continue
            combo_folder, date_str = m.group(1), m.group(2)
            text = zf.read(name).decode("utf-8", errors="replace")
            lines = text.split("\n")
            blank_idx = next((i for i, l in enumerate(lines) if l.strip() == ""), None)
            trades_lines = lines[:blank_idx] if blank_idx is not None else lines
            footer_lines = [l for l in (lines[blank_idx + 1:] if blank_idx is not None else []) if l.strip()]

            n = num_wins = 0
            gross_profit = gross_loss = invested = 0.0
            sym_agg = {}  # symbol -> [n, wins, gross_profit, gross_loss, invested]

            for line in trades_lines[1:]:
                if not line.strip():
                    continue
                parts = line.split(",")
                if len(parts) <= PNL_IDX:
                    continue
                try:
                    entry = float(parts[ENTRY_IDX])
                    shares = float(parts[SHARES_IDX])
                    pnl = float(parts[PNL_IDX])
                except ValueError:
                    continue
                symbol = parts[SYMBOL_IDX].strip()
                is_win = pnl > 0
                inv = entry * shares

                n += 1
                invested += inv
                if is_win:
                    gross_profit += pnl
                    num_wins += 1
                elif pnl < 0:
                    gross_loss += -pnl

                s = sym_agg.setdefault(symbol, [0, 0, 0.0, 0.0, 0.0])
                s[0] += 1
                s[4] += inv
                if is_win:
                    s[2] += pnl
                    s[1] += 1
                elif pnl < 0:
                    s[3] += -pnl

            footer_dict = {}
            if len(footer_lines) >= 2:
                headers = footer_lines[0].split(",")
                values = footer_lines[1].split(",")
                footer_dict = {h.strip(): _cast(v) for h, v in zip(headers, values)}

            net_pnl = gross_profit - gross_loss
            row = {"combo_folder": combo_folder, "date": date_str,
                   "num_trades": n, "num_wins": num_wins,
                   "gross_profit": gross_profit, "gross_loss": gross_loss,
                   "net_pnl_computed": net_pnl, "total_invested": invested}
            for k, v in footer_dict.items():
                row[f"footer_{k}" if k in FOOTER_RESULT_COLS else k] = v
            combo_day_rows.append(row)

            for symbol, (sn, swins, sgp, sgl, sinv) in sym_agg.items():
                combo_symbol_day_rows.append({
                    "combo_folder": combo_folder, "symbol": symbol, "date": date_str,
                    "num_trades": sn, "num_wins": swins,
                    "gross_profit": sgp, "gross_loss": sgl,
                    "net_pnl_computed": sgp - sgl, "total_invested": sinv})
    return combo_day_rows, combo_symbol_day_rows


def _finalize_days(df):
    if df.empty:
        return df
    df["date"] = pd.to_datetime(df["date"], format="%d-%m-%Y", errors="coerce")
    df["year"] = df["date"].dt.year
    df["quarter"] = df["date"].dt.to_period("Q").astype(str)
    df["combo_folder"] = df["combo_folder"].astype("category")
    if "symbol" in df.columns:
        df["symbol"] = df["symbol"].astype("category")
    for c in df.select_dtypes(include="float64").columns:
        df[c] = pd.to_numeric(df[c], downcast="float")
    for c in df.select_dtypes(include="int64").columns:
        df[c] = pd.to_numeric(df[c], downcast="integer")
    return df


def parse_zips(zip_paths, max_workers=None):
    if not zip_paths:
        raise FileNotFoundError(
            f"No .zip files found under {BASE_DIR} in folders {SOURCE_FOLDERS}. "
            "Check BASE_DIR/SOURCE_FOLDERS in Cell 1 and that Drive is mounted.")
    t0 = time.time()
    all_combo_rows, all_symbol_rows = [], []
    max_workers = max_workers or min(8, os.cpu_count() or 4)
    print(f"Parsing {len(zip_paths)} zip(s) with {max_workers} worker process(es)...")
    with ProcessPoolExecutor(max_workers=max_workers) as ex:
        futures = {ex.submit(_parse_one_zip, zp): zp for zp in zip_paths}
        for fut in as_completed(futures):
            zp = futures[fut]
            zt0 = time.time()
            combo_rows, symbol_rows = fut.result()
            all_combo_rows.extend(combo_rows)
            all_symbol_rows.extend(symbol_rows)
            print(f"  {os.path.basename(zp)}: {len(combo_rows)} combo-day rows ({time.time() - zt0:.1f}s)")
    print(f"Total parse time: {time.time() - t0:.1f}s, {len(all_combo_rows)} combo-day rows.")
    combo_days = _finalize_days(pd.DataFrame(all_combo_rows))
    combo_symbol_days = _finalize_days(pd.DataFrame(all_symbol_rows))
    return combo_days, combo_symbol_days


# ---- run parsing (only if no cache yet) ------------------------------------
combo_days_path = os.path.join(MASTER_DIR, "combo_days.parquet")
combo_symbol_days_path = os.path.join(MASTER_DIR, "combo_symbol_days.parquet")

if os.path.exists(combo_days_path) and os.path.exists(combo_symbol_days_path):
    print("Master cache found — loading from Parquet (delete files in", MASTER_DIR,
          "to force a re-parse).")
    combo_days = pd.read_parquet(combo_days_path)
    combo_symbol_days = pd.read_parquet(combo_symbol_days_path)
else:
    print(f"Scanning {len(SOURCE_FOLDERS)} folder(s) under {BASE_DIR}:")
    zip_paths = gather_zip_paths(BASE_DIR, SOURCE_FOLDERS)
    combo_days, combo_symbol_days = parse_zips(zip_paths)
    combo_days.to_parquet(combo_days_path, compression="snappy")
    combo_symbol_days.to_parquet(combo_symbol_days_path, compression="snappy")
    print(f"Master cache written to {MASTER_DIR}")

print(f"combo_days: {len(combo_days):,} rows, {combo_days['combo_folder'].nunique():,} combos, "
      f"{combo_days['date'].nunique():,} days, years {sorted(combo_days['year'].dropna().unique())}")


# %% [CELL 3] ================================================================
# COMBO IDS — short, stable labels (Combo_0001...) so every tab cross-references
# easily instead of long folder-name strings.
# =============================================================================
def assign_combo_ids(df):
    combos = sorted(df["combo_folder"].astype(str).unique())
    id_map = {c: f"Combo{i+1:03d}" for i, c in enumerate(combos)}  # Combo001, Combo002, ...
    combo_map = pd.DataFrame({"Combo ID": [id_map[c] for c in combos], "Combo Parameters": combos})
    return id_map, combo_map


id_map, combo_map = assign_combo_ids(combo_days)
combo_days["Combo ID"] = combo_days["combo_folder"].map(id_map)
combo_symbol_days["Combo ID"] = combo_symbol_days["combo_folder"].map(id_map)
print(f"{len(combo_map)} combos assigned IDs Combo001 .. {combo_map['Combo ID'].iloc[-1]}")


# %% [CELL 4] ================================================================
# METRICS ENGINE — same math as your original compute_full_metrics, usable
# for any grouping: [Combo ID], [Combo ID, year], [Combo ID, quarter] ...
# =============================================================================
def compute_full_metrics(df, group_cols, param_cols=PARAM_COLS):
    param_cols = [c for c in param_cols if c in df.columns]
    out_rows = []
    for keys, g in df.groupby(group_cols, observed=True):
        daily_pnl = g["net_pnl_computed"].values
        total_trades = g["num_trades"].sum()
        total_wins = g["num_wins"].sum()
        gross_profit, gross_loss = g["gross_profit"].sum(), g["gross_loss"].sum()
        total_invested = g["total_invested"].sum()
        hit_target = g["footer_hit_target"].sum() if "footer_hit_target" in g.columns else np.nan

        avg_pnl = daily_pnl.mean()
        std_pnl = daily_pnl.std(ddof=1) if len(daily_pnl) > 1 else np.nan
        median_pnl = np.median(daily_pnl)
        total_pnl = daily_pnl.sum()

        if len(daily_pnl) > 1 and std_pnl and std_pnl > 0:
            _, p_val = stats.ttest_1samp(daily_pnl, popmean=0)
        else:
            p_val = np.nan

        key_tuple = keys if isinstance(keys, tuple) else (keys,)
        row = dict(zip(group_cols, key_tuple))
        row.update({
            "Total (Pnl)": total_pnl, "Average (Pnl)": avg_pnl, "Standard Deviation": std_pnl,
            "Error (stdev/avg %)": (std_pnl / avg_pnl * 100) if avg_pnl else np.nan,
            "Median (Pnl)": median_pnl, "Total Trades": int(total_trades),
            "Hit Target %": (hit_target / total_trades * 100) if total_trades else np.nan,
            "Win rate %": (total_wins / total_trades * 100) if total_trades else np.nan,
            "Total Invested ($)": total_invested,
            "Profit Factor": (gross_profit / gross_loss) if gross_loss else np.inf,
            "Risk-Adjusted (Sharpe-like)": (avg_pnl / std_pnl) if std_pnl else np.nan,
            "Robustness (% Profitable Days)": (daily_pnl > 0).mean(),
            "p-value": p_val, "Days Tested": len(daily_pnl),
            "First Seen": g["date"].min(), "Last Seen": g["date"].max(),
        })
        if "Combo ID" in df.columns:
            for pc in param_cols:
                row[pc] = g[pc].iloc[0]
        out_rows.append(row)

    out = pd.DataFrame(out_rows)
    valid = out["p-value"].notna()
    corrected = pd.Series(np.nan, index=out.index)
    if valid.sum():
        _, pv, _, _ = multipletests(out.loc[valid, "p-value"], method="fdr_bh")
        corrected.loc[valid] = pv
    out["p-value (FDR-corrected)"] = corrected
    out["Return on Capital %"] = out["Total (Pnl)"] / out["Total Invested ($)"] * 100
    out["Expectancy ($/trade)"] = out["Total (Pnl)"] / out["Total Trades"]
    return out.sort_values("Total (Pnl)", ascending=False).reset_index(drop=True)


# Total (all years combined), Yearly, and Quarterly breakdowns — req. #6
total_metrics = compute_full_metrics(combo_days, ["Combo ID"])
yearly_metrics = compute_full_metrics(combo_days, ["Combo ID", "year"])
quarterly_metrics = compute_full_metrics(combo_days, ["Combo ID", "quarter"])
print(f"Total: {len(total_metrics)} combos | Yearly rows: {len(yearly_metrics)} | "
      f"Quarterly rows: {len(quarterly_metrics)}")


# %% [CELL 5] ================================================================
# BALANCE ("COMPOSITE") SCORE — per your definition: equal-weight blend of
# Robustness, Total PnL, Profit Factor, Expectancy, Average PnL, and
# (inverted) Error%. Applied at all three grains you asked for:
#   Total    -> one score per combo (ranked against every other combo, once)
#   Yearly   -> scored per combo per year, then rolled into Mean/Std/Min/Max
#               + Consistent Winner/Loser Score across years
#   Quarterly-> same roll-up, across quarters instead of years
# =============================================================================
def add_balance_score(period_metrics, period_col):
    df = period_metrics.copy()
    ranks = pd.DataFrame(index=df.index)
    for col, direction in BALANCE_COMPONENTS.items():
        vals = df[col].replace([np.inf, -np.inf], np.nan)
        period_best = vals.groupby(df[period_col]).transform("max" if direction == "higher" else "min")
        filled = df[col].where(np.isfinite(df[col]), period_best)
        if direction == "lower":
            filled = -filled
        ranks[col] = filled.groupby(df[period_col]).rank(pct=True, na_option="keep")
    df["Balance Score"] = ranks.mean(axis=1, skipna=True)
    return df


def consistency_from_periods(period_scored, suffix, id_col="Combo ID", score_col="Balance Score"):
    g = period_scored.groupby(id_col, observed=True)[score_col]
    out = g.agg(Mean="mean", Std="std", Min="min", Max="max", Periods_Tested="count").reset_index()
    out["Std"] = out["Std"].fillna(0.0)
    out[f"Consistent Winner Score ({suffix})"] = out["Mean"] - out["Std"]   # reliably good
    out[f"Consistent Loser Score ({suffix})"] = out["Mean"] + out["Std"]    # reliably bad
    return out.rename(columns={
        "Mean": f"Mean Composite Score ({suffix})", "Std": f"Std Composite Score ({suffix})",
        "Min": f"Min Composite Score ({suffix})", "Max": f"Max Composite Score ({suffix})",
        "Periods_Tested": f"Periods Tested ({suffix})"})


def exclude_low_session_outliers(df, count_col, k=IQR_K):
    """Drop combos with an abnormally LOW number of periods tested
    (IQR x1.5 low-fence rule) so thin-data combos can't fake consistency."""
    q1, q3 = df[count_col].quantile([0.25, 0.75])
    low_fence = q1 - k * (q3 - q1)
    kept = df[df[count_col] >= low_fence].copy()
    return kept, len(df) - len(kept), low_fence


# Total: one Balance Score per combo, ranked against every other combo once
total_scored = add_balance_score(total_metrics.assign(_all="ALL"), "_all").drop(columns="_all")
total_scored = total_scored.rename(columns={"Balance Score": "Balance Score (Total)"})

# Yearly and Quarterly: scored within each period, then rolled up into consistency stats
yearly_scored = add_balance_score(yearly_metrics, "year")
quarterly_scored = add_balance_score(quarterly_metrics, "quarter")
consistency_yearly = consistency_from_periods(yearly_scored, "Yearly")
consistency_quarterly = consistency_from_periods(quarterly_scored, "Quarterly")

combo_metrics = (total_scored
                  .merge(consistency_yearly, on="Combo ID", how="left")
                  .merge(consistency_quarterly, on="Combo ID", how="left")
                  .merge(combo_map, on="Combo ID", how="left"))

# For the Top-30 consistency tabs, exclude combos with an abnormally low
# number of YEARS tested (the granularity you compare combos across).
combo_metrics_kept, n_dropped, low_fence = exclude_low_session_outliers(combo_metrics, "Periods Tested (Yearly)")
print(f"Excluded {n_dropped} combo(s) as low-session outliers (fewer than {low_fence:.1f} years tested).")


# %% [CELL 6] ================================================================
# RANKING TAB — one row per combo, Combo ID first, matches the column layout
# you specified.
# =============================================================================
RANKING_COLS = [
    "Combo ID", "Combo Parameters", "Balance Score (Total)",
    "Mean Composite Score (Yearly)", "Std Composite Score (Yearly)",
    "Min Composite Score (Yearly)", "Max Composite Score (Yearly)",
    "Consistent Winner Score (Yearly)", "Consistent Loser Score (Yearly)",
    "Periods Tested (Yearly)",
    "Mean Composite Score (Quarterly)", "Std Composite Score (Quarterly)",
    "Min Composite Score (Quarterly)", "Max Composite Score (Quarterly)",
    "Consistent Winner Score (Quarterly)", "Consistent Loser Score (Quarterly)",
    "Periods Tested (Quarterly)",
    "Total (Pnl)", "Average (Pnl)", "Standard Deviation", "Error (stdev/avg %)",
    "Median (Pnl)", "Total Trades", "Hit Target %", "Win rate %", "Total Invested ($)",
    "Profit Factor", "Risk-Adjusted (Sharpe-like)", "Robustness (% Profitable Days)",
    "p-value", "p-value (FDR-corrected)", "Return on Capital %", "Expectancy ($/trade)",
]
ranking_tab = combo_metrics[[c for c in RANKING_COLS if c in combo_metrics.columns]] \
    .sort_values("Consistent Winner Score (Yearly)", ascending=False).reset_index(drop=True)
print(ranking_tab.head(10).to_string(index=False))


# %% [CELL 7] ================================================================
# TOP 30 WINNERS / LOSERS BY CONSISTENCY (low-session outliers excluded),
# plus the "Significant" variants filtered to FDR-corrected p <= 0.05.
# =============================================================================
top_winners = combo_metrics_kept.sort_values("Consistent Winner Score (Yearly)", ascending=False).head(TOP_N)
top_losers = combo_metrics_kept.sort_values("Consistent Loser Score (Yearly)", ascending=True).head(TOP_N)

sig = combo_metrics_kept[combo_metrics_kept["p-value (FDR-corrected)"] <= FDR_ALPHA]
top_winners_sig = sig.sort_values("Consistent Winner Score (Yearly)", ascending=False).head(TOP_N)
top_losers_sig = sig.sort_values("Consistent Loser Score (Yearly)", ascending=True).head(TOP_N)

DETAIL_COLS = ["Combo ID", "Combo Parameters", "Periods Tested (Yearly)", "Mean Composite Score (Yearly)",
               "Std Composite Score (Yearly)", "Consistent Winner Score (Yearly)",
               "Consistent Loser Score (Yearly)",
               "Total (Pnl)", "Robustness (% Profitable Days)", "p-value (FDR-corrected)",
               "Return on Capital %"]
top_winners = top_winners[[c for c in DETAIL_COLS if c in top_winners.columns]]
top_losers = top_losers[[c for c in DETAIL_COLS if c in top_losers.columns]]
top_winners_sig = top_winners_sig[[c for c in DETAIL_COLS if c in top_winners_sig.columns]]
top_losers_sig = top_losers_sig[[c for c in DETAIL_COLS if c in top_losers_sig.columns]]
print(f"Winners: {len(top_winners)}, Losers: {len(top_losers)}, "
      f"Significant Winners: {len(top_winners_sig)}, Significant Losers: {len(top_losers_sig)}")


# %% [CELL 8] ================================================================
# PARAMETER PERFORMANCE / BEST / WORST / SENSITIVITY — how much does each
# knob move the needle, holding everything else as tested.
# =============================================================================
def parameter_performance(combo_metrics_total, param_cols=PARAM_COLS,
                           value_cols=("Balance Score (Total)", "Total (Pnl)", "Win rate %",
                                       "Robustness (% Profitable Days)", "Return on Capital %")):
    param_cols = [c for c in param_cols if c in combo_metrics_total.columns]
    value_cols = [c for c in value_cols if c in combo_metrics_total.columns]
    long_rows = []
    for pc in param_cols:
        g = combo_metrics_total.groupby(pc, observed=True)[value_cols].mean().reset_index()
        g.insert(0, "Parameter", pc)
        g = g.rename(columns={pc: "Value"})
        g["Combos"] = combo_metrics_total.groupby(pc, observed=True).size().values
        long_rows.append(g)
    perf = pd.concat(long_rows, ignore_index=True) if long_rows else pd.DataFrame()

    sens_rows = []
    for pc in param_cols:
        means = combo_metrics_total.groupby(pc, observed=True)["Balance Score (Total)"].mean()
        if len(means) > 1:
            sens_rows.append({"Parameter": pc,
                               "Sensitivity (Balance Score range)": means.max() - means.min(),
                               "Best Value": means.idxmax(), "Worst Value": means.idxmin()})
    sensitivity = (pd.DataFrame(sens_rows)
                   .sort_values("Sensitivity (Balance Score range)", ascending=False)
                   .reset_index(drop=True)) if sens_rows else pd.DataFrame()

    best = (perf.sort_values("Balance Score (Total)", ascending=False)
            .groupby("Parameter", observed=True).head(1).reset_index(drop=True)) if len(perf) else perf
    worst = (perf.sort_values("Balance Score (Total)", ascending=True)
             .groupby("Parameter", observed=True).head(1).reset_index(drop=True)) if len(perf) else perf
    return perf, best, worst, sensitivity


param_perf, param_best, param_worst, param_sensitivity = parameter_performance(combo_metrics)
print(param_sensitivity.to_string(index=False) if len(param_sensitivity) else "No PARAM_COLS found in data.")


# %% [CELL 9] ================================================================
# SHORTLIST — combos that pass ALL of: FDR-significant, robust (>=55% days
# profitable), ranked by Return on Capital %. "Which combo can I trust."
# =============================================================================
shortlist = combo_metrics[
    (combo_metrics["p-value (FDR-corrected)"] <= FDR_ALPHA) &
    (combo_metrics["Robustness (% Profitable Days)"] >= ROBUSTNESS_MIN)
].sort_values("Return on Capital %", ascending=False).reset_index(drop=True)
shortlist = shortlist[[c for c in RANKING_COLS if c in shortlist.columns]]
print(f"Shortlist: {len(shortlist)} combo(s) pass all filters.")


# %% [CELL 10] ===============================================================
# SYMBOL STATS — one row per symbol (conflating every combo/day). Answers
# "is this symbol worth keeping in BT_SYMBOLS at all".
# =============================================================================
symbol_stats = compute_full_metrics(
    combo_symbol_days.rename(columns={"symbol": "Symbol"}), ["Symbol"])

SYMBOL_COLS = ["Symbol", "Total (Pnl)", "Average (Pnl)", "Win rate %", "Profit Factor",
               "Robustness (% Profitable Days)", "Total Trades", "Return on Capital %",
               "p-value (FDR-corrected)"]
symbol_stats = symbol_stats[[c for c in SYMBOL_COLS if c in symbol_stats.columns]] \
    .sort_values("Total (Pnl)", ascending=False).reset_index(drop=True)
print(f"Symbol Stats: {len(symbol_stats)} symbols.")


# %% [CELL 11] ===============================================================
# TOTAL / YEARLY / QUARTERLY TABS — long format, one row per (combo, period),
# each carrying the full metric set (not a trimmed subset).
# =============================================================================
FULL_METRIC_COLS = [
    "Total (Pnl)", "Average (Pnl)", "Standard Deviation", "Error (stdev/avg %)",
    "Median (Pnl)", "Total Trades", "Hit Target %", "Win rate %", "Total Invested ($)",
    "Profit Factor", "Risk-Adjusted (Sharpe-like)", "Robustness (% Profitable Days)",
    "p-value", "p-value (FDR-corrected)", "Return on Capital %", "Expectancy ($/trade)",
]

yearly_tab = yearly_scored.merge(combo_map, on="Combo ID")[
    ["Combo ID", "Combo Parameters", "year"] +
    [c for c in FULL_METRIC_COLS if c in yearly_scored.columns] + ["Balance Score"]
].sort_values(["year", "Total (Pnl)"], ascending=[True, False]).reset_index(drop=True)

quarterly_tab = quarterly_scored.merge(combo_map, on="Combo ID")[
    ["Combo ID", "Combo Parameters", "quarter"] +
    [c for c in FULL_METRIC_COLS if c in quarterly_scored.columns] + ["Balance Score"]
].sort_values(["quarter", "Total (Pnl)"], ascending=[True, False]).reset_index(drop=True)

total_tab = total_scored.merge(combo_map, on="Combo ID")[
    ["Combo ID", "Combo Parameters"] +
    [c for c in FULL_METRIC_COLS if c in total_scored.columns] + ["Balance Score (Total)"]
].sort_values("Total (Pnl)", ascending=False).reset_index(drop=True)


# %% [CELL 12] ===============================================================
# GUIDE TAB — plain-English glossary of every tab and metric, so this is
# self-explanatory to anyone opening the workbook cold.
# =============================================================================
guide_rows = [
    ("Tabs", "Guide", "This sheet. What every tab and column means."),
    ("Tabs", "Ranking", "One row per combo (Combo ID + Combo Parameters), full metric set at "
                         "Total/Yearly/Quarterly granularity, sorted by Consistent Winner Score (Yearly). "
                         "Your main leaderboard."),
    ("Tabs", "Top Winners", f"Top {TOP_N} combos by Consistent Winner Score (Yearly). Combos tested in "
                             "an abnormally low number of years (IQR x1.5 low-outlier rule) are excluded "
                             "so thin data can't fake consistency."),
    ("Tabs", "Top Losers", f"Mirror of Top Winners: top {TOP_N} combos by Consistent Loser Score (Yearly) "
                            "-- reliably bad, not just unlucky once."),
    ("Tabs", "Top Winners (Significant)", "Same as Top Winners, additionally filtered to combos whose "
                                           f"FDR-corrected p-value <= {FDR_ALPHA}. With very little data "
                                           "(e.g. a single test year) this can legitimately come back empty "
                                           "-- it means nothing has cleared statistical significance yet, "
                                           "not that anything is broken."),
    ("Tabs", "Top Losers (Significant)", "Same idea as Top Winners (Significant), for the loser side."),
    ("Tabs", "Shortlist", "Combos that pass ALL of: FDR-significant (p <= " + str(FDR_ALPHA) + "), "
                           f"robust (>= {int(ROBUSTNESS_MIN*100)}% of days profitable), ranked by Return "
                           "on Capital %. The answer to 'which combo should I actually trust', not just "
                           "'which combo has the highest average'."),
    ("Tabs", "Parameter Performance", "For each parameter (e.g. orb_minutes) and each value it was tested "
                                       "with, the average Balance Score / PnL / Win rate / Robustness / ROC% "
                                       "across every combo that used that value."),
    ("Tabs", "Best / Worst Parameters", "For each parameter, the single value with the highest / lowest "
                                         "average Balance Score."),
    ("Tabs", "Parameter Sensitivity", "How much each parameter matters: the spread (max-min) of average "
                                       "Balance Score across that parameter's tested values. A high number "
                                       "means changing this knob swings performance a lot; a low number "
                                       "means it barely matters."),
    ("Tabs", "Total", "One row per combo, metrics computed over ALL data combined (every year/quarter you "
                       "fed in)."),
    ("Tabs", "Yearly", "One row per (combo, year) -- lets you see a combo's performance year by year."),
    ("Tabs", "Quarterly", "One row per (combo, quarter) -- finer-grained version of Yearly."),
    ("Tabs", "Symbol Stats", "One row per symbol, combining every combo and day together. Answers 'is this "
                              "symbol worth keeping at all', not 'how does it do under my best combo'."),
    ("Combo Identity", "Combo ID", "Short stable label (Combo001, Combo002, ...) used everywhere instead "
                                    "of the long parameter string, so tabs cross-reference easily."),
    ("Combo Identity", "Combo Parameters", "The raw combo folder name / identifier -- effectively the "
                                            "parameter combination that produced this row (e.g. which "
                                            "orb_minutes, filters, multipliers were used). Kept narrow in "
                                            "these sheets; widen the column if you need to read it in full."),
    ("Composite Score", "Balance Score", "0-1 blended score (equal weight) of Robustness, Total PnL, Profit "
                                          "Factor, Expectancy, Average PnL, and (inverted) Error %. Each "
                                          "component is converted to a percentile rank against the combo's "
                                          "PEERS IN THE SAME PERIOD before averaging, so it's not dominated "
                                          "by one spectacular metric or by a large-dollar-value outlier."),
    ("Composite Score", "Mean/Std/Min/Max Composite Score (Yearly / Quarterly)", "The Balance Score's mean, "
                                          "standard deviation, min and max across a combo's tested years or "
                                          "quarters -- i.e. how consistent that combo's blended performance "
                                          "is over time."),
    ("Composite Score", "Consistent Winner Score", "Mean - Std of the Balance Score across periods. "
                                          "Penalizes combos whose performance swings a lot; rewards combos "
                                          "that are reliably good every period."),
    ("Composite Score", "Consistent Loser Score", "Mean + Std of the Balance Score across periods. The "
                                          "mirror of Consistent Winner Score: ranks low for combos that are "
                                          "reliably bad, not just occasionally terrible."),
    ("Composite Score", "Periods Tested (Yearly / Quarterly)", "How many distinct years / quarters this "
                                          "combo has data for. Low values are the ones excluded from Top "
                                          "Winners/Losers by the IQR outlier rule."),
    ("Core Metrics", "Total (Pnl)", "Sum of net PnL across every day in the period."),
    ("Core Metrics", "Average (Pnl)", "Mean daily net PnL."),
    ("Core Metrics", "Standard Deviation", "Standard deviation of daily net PnL (day-to-day volatility)."),
    ("Core Metrics", "Error (stdev/avg %)", "Standard Deviation / Average PnL, as a percent -- a rough "
                                             "'noise relative to signal' gauge. Lower is more consistent."),
    ("Core Metrics", "Median (Pnl)", "Median daily net PnL -- less sensitive to a single huge day than the "
                                      "average."),
    ("Core Metrics", "Total Trades", "Total number of individual trades across the period."),
    ("Core Metrics", "Hit Target %", "Percent of days the footer reported the target as hit."),
    ("Core Metrics", "Win rate %", "Percent of individual trades that were profitable."),
    ("Core Metrics", "Total Invested ($)", "Sum of (entry price x shares) across every trade -- capital put "
                                            "to work, not capital at risk."),
    ("Core Metrics", "Profit Factor", "Gross profit / gross loss. >1 means more won than lost; treated as "
                                       "infinite when there were zero losing trades."),
    ("Core Metrics", "Risk-Adjusted (Sharpe-like)", "Average PnL / Standard Deviation of daily PnL -- higher "
                                                      "is better return per unit of day-to-day volatility."),
    ("Core Metrics", "Robustness (% Profitable Days)", "Percent of days with positive net PnL."),
    ("Core Metrics", "p-value", "One-sample t-test that daily PnL is different from zero. Lower = more "
                                 "confidence the average PnL isn't just noise."),
    ("Core Metrics", "p-value (FDR-corrected)", "p-value adjusted (Benjamini-Hochberg) for the fact that "
                                                 "many combos were tested at once -- the honest number to "
                                                 "filter on, not the raw p-value."),
    ("Core Metrics", "Return on Capital %", "Total (Pnl) / Total Invested ($) x 100."),
    ("Core Metrics", "Expectancy ($/trade)", "Total (Pnl) / Total Trades -- average profit per trade."),
    ("Config used", "FDR significance threshold", f"{FDR_ALPHA}"),
    ("Config used", "Robustness minimum (Shortlist)", f"{ROBUSTNESS_MIN} ({int(ROBUSTNESS_MIN*100)}% of days profitable)"),
    ("Config used", "IQR low-outlier fence multiplier", f"{IQR_K}"),
    ("Config used", "Top N (Winners/Losers)", f"{TOP_N}"),
]
guide_tab = pd.DataFrame(guide_rows, columns=["Section", "Item", "Explanation"])


# %% [CELL 13] ===============================================================
# WRITE THE ANALYSIS WORKBOOK — Combo-Day Detail is intentionally NOT
# included (it lives only in the Parquet master cache under MASTER_DIR);
# at this scale it would blow past Excel's ~1.05M row limit for no
# analytical benefit. Every sheet gets: a filterable, coloured header row,
# a frozen header, and auto-fit column widths -- except "Combo Parameters",
# which is kept narrow (it's the long raw combo-folder string) so it
# doesn't blow out the sheet; widen it manually or double-click the column
# border to see it in full.
# =============================================================================
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

HEADER_FILL = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
HEADER_FONT = Font(color="FFFFFF", bold=True)
NARROW_COLS = {"Combo Parameters"}
NARROW_WIDTH = 16
MAX_WIDTH = 32


def style_sheet(writer, sheet_name, df):
    ws = writer.sheets[sheet_name]
    if df.empty:
        return
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions
    for idx, col in enumerate(df.columns, start=1):
        cell = ws.cell(row=1, column=idx)
        cell.fill = HEADER_FILL
        cell.font = HEADER_FONT
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        col_letter = get_column_letter(idx)
        if col in NARROW_COLS:
            ws.column_dimensions[col_letter].width = NARROW_WIDTH
        else:
            sample = df[col].astype(str).head(1000)
            max_len = max([len(str(col))] + [len(v) for v in sample])
            ws.column_dimensions[col_letter].width = min(max(10, max_len + 2), MAX_WIDTH)


SHEETS = [
    ("Guide", guide_tab),
    ("Ranking", ranking_tab),
    ("Top Winners", top_winners),
    ("Top Losers", top_losers),
    ("Top Winners (Significant)", top_winners_sig),
    ("Top Losers (Significant)", top_losers_sig),
    ("Shortlist", shortlist),
    ("Parameter Performance", param_perf),
    ("Best Parameters", param_best),
    ("Worst Parameters", param_worst),
    ("Parameter Sensitivity", param_sensitivity),
    ("Total", total_tab),
    ("Yearly", yearly_tab),
    ("Quarterly", quarterly_tab),
    ("Symbol Stats", symbol_stats),
]

with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:
    for sheet_name, df in SHEETS:
        df.to_excel(writer, sheet_name=sheet_name, index=False)
        style_sheet(writer, sheet_name, df)

print(f"Workbook written: {OUT_XLSX}")
print(f"Master Parquet cache: {MASTER_DIR} (combo_days.parquet, combo_symbol_days.parquet)")

try:
    from google.colab import files
    files.download(OUT_XLSX)
except ImportError:
    pass